# Beyond the Scoreboard

Giuseppe Bagnara & Giovanni Passamonti

## The dataset

The dataset we used for our analysis is an Open Access Repository related to the paper "The FIFA World Cup 2026 Master Dataset: A Normalized 3NF 
Relational Benchmark of the Expanded 48\-Team International Football Tournament" \(Islam, M\. M\., 2026\)\. 

This dataset was obtained combining official FIFA bulletins, SofaScore technical summaries and TransferMarkt bio\-metrics\. Data contained is related to all the 104 matches of the 48\-team tournament, featuring also 1248 registered players, 2704 minute\-by\-minute lineup records, tactical team statistics, expected goals \(xG\) and pre\-engineered predictive feature matrices\. All the data was later normalized into 12 relational tables\.

## The hypothesis

Do xGs, ball possession and shots describe a match? This is what we'll call "case 1"\. The three values, taken independently, are unable to provide us with a clear interpretation of a match and the consequences on the field\. What we'll try to achieve is to see if there is an actual correlation between dominating a game and earning a victory\.

Are there unexpected matches' results? This will be defined as "case 2"\. After we're able to describe a match, we want to verify if there exists a way to classify the matches that showed a result different from the expected outcome and on the statistics, based on the metrics we were able to calculate\.

## The data

In [1]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

third_performance_variable = os.environ.get("third_performance_variable", "sot")

What we'll call third\_performance\_variable the shots on target made by a team, in order to see how they were actually looking for a goal\.

In [2]:
matches = pd.read_csv("/work/match_prediction_features.csv")
team_stats = pd.read_csv("/work/match_team_stats.csv").set_index(["match_id", "team_id"])
cols = ["possession_pct", "total_shots", "shots_on_target"]

We read the match\_prediction\_features and match\_team\_stats CSVs to get the values we'll use later on\.

In [3]:
for s in ["home", "away"]:
    i = pd.MultiIndex.from_arrays([matches.match_id, matches[f"{s}_team_id"]])
    matches = pd.concat([matches, team_stats.reindex(i)[cols].reset_index(drop=True).add_prefix(s + "_")], axis=1)

matches

,match_id,date,kickoff_time_utc,stage_id,is_knockout,home_team_id,home_team_name,home_fifa_code,home_confederation,away_team_id,...,result_type,home_xg,away_xg,match_result,home_possession_pct,home_total_shots,home_shots_on_target,away_possession_pct,away_total_shots,away_shots_on_target
0,1,2026-06-11,19:00,1,0,1,Mexico,MEX,CONCACAF,2,...,Regular,1.84,0.52,H,57,16,4,43,3,2
1,2,2026-06-11,23:00,1,0,3,South Korea,KOR,AFC,4,...,Regular,1.45,1.12,H,54,12,5,46,10,3
2,3,2026-06-12,19:00,1,0,5,Canada,CAN,CONCACAF,6,...,Regular,1.35,0.98,D,52,13,4,29,8,3
3,4,2026-06-12,23:00,1,0,13,USA,USA,CONCACAF,14,...,Regular,2.76,0.88,H,59,16,6,28,9,1
4,5,2026-06-13,15:00,1,0,7,Qatar,QAT,AFC,8,...,Regular,0.78,1.54,D,28,6,3,58,26,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,100,2026-07-12,01:00,4,1,37,Argentina,ARG,CONMEBOL,8,...,AET,2.00,0.53,H,63,18,8,37,8,2
100,101,2026-07-14,20:00,5,1,33,France,FRA,UEFA,29,...,Regular,0.30,1.63,A,41,8,2,59,15,6
101,102,2026-07-15,20:00,5,1,45,England,ENG,UEFA,37,...,Regular,0.54,1.80,A,47,10,4,53,13,5
102,103,2026-07-18,22:00,6,1,33,France,FRA,UEFA,45,...,Regular,2.88,2.88,A,50,18,9,50,21,11


Now, we've extracted all the matches creating a dataframe of them\.

In [4]:
# Ball possession normalization
tot = matches.home_possession_pct + matches.away_possession_pct
matches["home_pos"] = matches.home_possession_pct / tot * 100
matches["away_pos"] = 100 - matches.home_pos

Because some of the matches didn't sum up to 100%, we normalized the ball possession matches that don't have matches\.home\_possession\_pct \+ matches\.away\_possession\_pct = 100\.

In [5]:
def side(team, op):
    return pd.DataFrame({
        "match_id": matches.match_id, 
        "team": matches[f"{team}_team_name"], 
        "opponent": matches[f"{op}_team_name"],
        "xg_delta": matches[f"{team}_xg"] - matches[f"{op}_xg"],
        "poss_delta": 2 * matches[f"{team}_pos"] - 100, # expressed in %
        "sot_delta": matches[f"{team}_shots_on_target"] - matches[f"{op}_shots_on_target"],
        "shots_delta": matches[f"{team}_total_shots"] - matches[f"{op}_total_shots"],
        "goal_delta": matches[f"{team}_score"] - matches[f"{op}_score"]
    })

By defining the side\(team, op\) function, we create a dataframe that represents the vectorialization of a match from each team's perspective\. This way, we'll obtain 208 vectors that we'll store into the dataframe variable\.

match\_id: match's id in the dataset; team: the team's name; opponent: the opponent of the match taken into exam; xg\_delta: the difference between the xG of the team vs\. the opponent's one; poss\_delta: expressed in percentage points, represents the difference in ball possession in the match involving the team and the opponent; sot\_delta: the difference between shots on target in the match; shots\_delta: the difference of total shots during a match; goal\_delta: the difference between goals made during a match\.

In [6]:
dataframe = pd.concat([side("home", "away"), side("away", "home")], ignore_index=True)
dataframe["result"] = np.sign(dataframe.goal_delta).map({1: "Win", 0: "Draw", -1: "Loss"})
THIRDCOL = {"sot": "sot_delta", "shots": "shots_delta"}[third_performance_variable]
F1 = ["xg_delta", "poss_delta", THIRDCOL, "goal_delta"]

We ended up the preparation of the data by running side\(\) per team, in order to obtain 2 dataframes from the same match to represent each perspective; to add further data, we also inserted a "result" value that represents the final result of the match played, also calculating the THIRDCOL value and a set of features we'll call F1\.

### CASE 1: k\-means, with k = 4 classes

Now that we were able to define a vector for each team based on the performance of a match, we'll now calculate some other values we'll need, such as a standard scaler that fits the dataframe using the features in F1, and defining a k\-mean with 4 classes\. Later on we'll define those\.

In [7]:
Z = StandardScaler().fit_transform(dataframe[F1])
kmean = KMeans(4, n_init=100, random_state=0).fit(Z)

RES = {}
RES["silhouette_k4"] = float(silhouette_score(Z, kmean.labels_))
dataframe["raw"] = kmean.labels_
perf = dataframe.groupby("raw")[["xg_delta", "poss_delta", THIRDCOL]].apply(lambda g: (g / dataframe[g.columns].std()).mean().mean()) # performance score for each cluster
gd = dataframe.groupby("raw").goal_delta.mean()
lo = perf.idxmin() # dominated team
hi = perf.idxmax() # dominating team
mid = [c for c in perf.index if c not in (lo, hi)] # equilibrated
if gd[mid[0]] > gd[mid[1]]:
    eq_win = mid[0]
    eq_loss = mid[1]
else:
    eq_win = mid[1]
    eq_loss = mid[0]

The four classes we chose to describe a match are the following\. "Dominating", describes a team that had a significant advantage and better statistics than the opponent; "Equilibrated, best", out of the two teams there wasn't a big gap in performance, but the team did slightly better than the opponent; "Equilibrated, worst", the team that was on par with the opponent, but did worse in terms of performance; "Dominated", a team that had significantly less the control of the match\.

In [8]:
NAME = {hi: "Dominating", lo: "Dominated", eq_win: "Equilibrated, best", eq_loss: "Equilibrated, worst"}
dataframe["kmean_result"]= dataframe.raw.map(NAME)
ORDER = ["Dominating", "Equilibrated, best", "Equilibrated, worst", "Dominated"]
ColourCodes = {
    "Dominating": "#08519C", 
    "Equilibrated, best" : "#2E86C1",
    "Equilibrated, worst" : "#F28E00",
    "Dominated" : "#A63603"
}
center = dataframe.groupby("kmean_result")[F1].mean().loc[ORDER]
comp = dataframe.groupby("kmean_result").result.value_counts(normalize=True).unstack().reindex(columns=["Win", "Draw", "Loss"]).fillna(0).loc[ORDER]
comp["n"] = dataframe.groupby("kmean_result").size().loc[ORDER]
print(center)
print(comp)

                     xg_delta  poss_delta  sot_delta  goal_delta
kmean_result                                                    
Dominating           1.989167   24.495770   6.111111    2.722222
Equilibrated, best   0.476029   15.321752   2.147059    0.676471
Equilibrated, worst -0.476029  -15.321752  -2.147059   -0.676471
Dominated           -1.989167  -24.495770  -6.111111   -2.722222
result                    Win      Draw      Loss   n
kmean_result                                         
Dominating           0.944444  0.055556  0.000000  36
Equilibrated, best   0.558824  0.323529  0.117647  68
Equilibrated, worst  0.117647  0.323529  0.558824  68
Dominated            0.000000  0.055556  0.944444  36


After obtaining our values, we save those in the RES matrix\.

In [9]:
RES["case1"] = {
    c: {
        "n": int(comp.loc[c, "n"]),
        "Win": float(comp.loc[c, "Win"]),
        "Draw": float(comp.loc[c, "Draw"]),
        "Loss": float(comp.loc[c, "Loss"]), 
        **{
            k: float(v) for k, v in center.loc[c].items()
        }
    }
    for c in ORDER
}

### CASE 2: k\-means, with k = 2 classes

Now, we'll move on to understand if and how we can define an unexpected match\. To make it possible, we calculate a performance\_index that represents the standard deviation of the performance, and a goal\_index that is also the standard deviation of the difference of goals\.

In [11]:
home_perspective = dataframe.iloc[:len(matches)].reset_index(drop=True) # one row for each match
sc = StandardScaler().fit(dataframe[["xg_delta", "poss_delta", THIRDCOL]])
performance_all = pd.Series(sc.transform(dataframe[["xg_delta", "poss_delta", THIRDCOL]]).mean(1))
performance_index = performance_all.std() # standard deviation of performance
goal_index = dataframe.goal_delta.std() # standard deviation of delta goals
home_perspective

,match_id,team,opponent,xg_delta,poss_delta,sot_delta,shots_delta,goal_delta,result,raw,kmean_result
0,1,Mexico,South Africa,1.32,14.000000,2,13,2,Win,2,"Equilibrated, best"
1,2,South Korea,Czechia,0.33,8.000000,2,2,1,Win,2,"Equilibrated, best"
2,3,Canada,Bosnia and Herzegovina,0.37,28.395062,1,5,0,Draw,2,"Equilibrated, best"
3,4,USA,Paraguay,1.88,35.632184,5,7,3,Win,1,Dominating
4,5,Qatar,Switzerland,-0.76,-34.883721,-4,-20,0,Draw,0,"Equilibrated, worst"
...,...,...,...,...,...,...,...,...,...,...,...
99,100,Argentina,Switzerland,1.47,26.000000,6,10,2,Win,1,Dominating
100,101,France,Spain,-1.33,-18.000000,-4,-7,-2,Loss,3,Dominated
101,102,England,Argentina,-1.26,-6.000000,-1,-3,-1,Loss,0,"Equilibrated, worst"
102,103,France,England,0.00,0.000000,-2,-3,-2,Loss,0,"Equilibrated, worst"


To determine whether or not the match presents an unexpected result, we'll compute the "surprise" factor: this represents the strenght of the better side\. Also, we define "draw\_dom" to express a dominated draw\.

In [12]:
home_perspective["perf_idx"] = pd.Series(sc.transform(home_perspective[["xg_delta", "poss_delta", THIRDCOL]]).mean(1)) # performance index
home_perspective["goal_normalized"] = home_perspective.goal_delta / goal_index
home_perspective["aligned"] = home_perspective.perf_idx * np.sign(home_perspective.goal_delta)
# >0: winner had the better performance
# <0: winner played worse
# =0: draw
home_perspective["draw_dom"] = np.where( # dominated draw
    home_perspective.goal_delta == 0, 
    home_perspective.perf_idx.abs(), 
    0.0
) 
home_perspective["surprise"] = np.where( # strenght of the better side
    home_perspective.goal_delta == 0,
    home_perspective.perf_idx.abs(),
    np.maximum(-home_perspective.aligned, 0)
)

Now, we'll compute the values of the second case, using a k\-mean with k=2, representing the two classes we chose: "Expected" and "Unexpected"\.

In [13]:
Z2 = home_perspective[["surprise"]].values
kmean2 = KMeans(2, n_init=100, random_state=0).fit(Z2)
RES["silhouette_k2"] = float(silhouette_score(Z2, kmean2.labels_))
home_perspective["raw2"] = kmean2.labels_
exp_c = home_perspective.groupby("raw2").surprise.mean().idxmin()
home_perspective["case2"] = np.where(home_perspective.raw2 == exp_c, "Expected", "Unexpected")
RES["case2_share_expected"] = float((home_perspective.case2 == "Expected").mean())
RES["case2_n"] = home_perspective.case2.value_counts().to_dict()
RES["case2_threshold_surprise"] = float((
    home_perspective[home_perspective.case2 == "Expected"].surprise.max()+ home_perspective[home_perspective.case2 == "Unexpected"].surprise.min())/2
)
RES["sign_agreement_non_draws"] = float((np.sign(home_perspective.perf_idx) == np.sign(home_perspective.goal_normalized))[home_perspective.goal_delta !=0].mean())
meta = matches[["match_id", "home_team_name", "away_team_name", "home_score", "away_score", "result_type", "stage_id"]]
home_perspective = home_perspective.merge(meta[["match_id", "home_team_name", "away_team_name", "home_score", "away_score", "result_type", "stage_id"]], on="match_id")
dataframe = dataframe.merge(home_perspective[["match_id", "case2"]], on="match_id")
unx = dataframe.groupby("kmean_result").case2.apply(lambda s: (s == "Unexpected").mean()).loc[ORDER]
RES["Unexpected_share_by_kmean"] = unx.to_dict()
print(home_perspective.groupby("case2")[["perf_idx", "goal_normalized", "aligned", "draw_dom", "surprise"]].mean().round(2));

            perf_idx  goal_normalized  aligned  draw_dom  surprise
case2                                                             
Expected        0.18             0.27     0.63      0.04      0.06
Unexpected      0.42             0.00    -0.15      0.81      0.97
{'silhouette_k4': 0.2628025393224978, 'case1': {'Dominating': {'n': 36, 'Win': 0.9444444444444444, 'Draw': 0.05555555555555555, 'Loss': 0.0, 'xg_delta': 1.9891666666666667, 'poss_delta': 24.49576990458097, 'sot_delta': 6.111111111111111, 'goal_delta': 2.7222222222222223}, 'Equilibrated, best': {'n': 68, 'Win': 0.5588235294117647, 'Draw': 0.3235294117647059, 'Loss': 0.11764705882352941, 'xg_delta': 0.4760294117647059, 'poss_delta': 15.32175179854468, 'sot_delta': 2.1470588235294117, 'goal_delta': 0.6764705882352942}, 'Equilibrated, worst': {'n': 68, 'Win': 0.11764705882352941, 'Draw': 0.3235294117647059, 'Loss': 0.5588235294117647, 'xg_delta': -0.47602941176470587, 'poss_delta': -15.32175179854468, 'sot_delta': -2.147058823

In [14]:
home_perspective["label"] = home_perspective.apply(lambda r: f"{r.team} {r.home_score}-{r.away_score} {r.opponent}" + (" (pens)" if r.result_type == "Penalties" else ""), axis=1)
top = home_perspective[home_perspective.case2 == "Unexpected"].sort_values("surprise", ascending=False).head(10)
home_perspective.drop(columns=["raw", "raw2"], errors="ignore").to_csv("/work/case2_matches_expected_vs_unexpected.csv", index=False)
dataframe.drop(columns=["raw"]).to_csv("/work/case1_team_match_classes.csv", index=False)
json.dump(RES, open("/work/beyond_results.json", "w"), indent=1, default=float)

## Visualization

In [3]:
MK = {"Dominating": "^", "Equilibrated, best": "o", "Equilibrated, worst": "s", "Dominated": "v"}
OUT = {"Win": "#0072B2", "Draw": "#6B6B6B", "Loss": "#D55E00"}                     
EXPC, UNXC = "#5E6B78", "#E6007E"
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14, "xtick.labelsize": 13, "ytick.labelsize": 13,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold", "axes.linewidth": 1.4})
fig = plt.figure(figsize=(19.2, 10.8))
gt = GridSpec(1, 6, figure=fig, left=.07, right=.98, top=.89, bottom=.55, wspace=1.75)
gb = GridSpec(1, 2, figure=fig, left=.07, right=.98, top=.38, bottom=.08, wspace=.95)
a1, a2, a3 = fig.add_subplot(gt[0, 0:2]), fig.add_subplot(gt[0, 2:4]), fig.add_subplot(gt[0, 4:6])
a4, a5 = fig.add_subplot(gb[0, 0]), fig.add_subplot(gb[0, 1])

NameError: name 'plt' is not defined

In [ ]:
for c in ORDER:  # 1. clusters, direct labels instead of a legend
    g = dataframe[dataframe["kmean_result"] == c]
    a1.scatter(
        g["xg_delta"],
        g["poss_delta"],
        s=85,
        c=ColourCodes[c],
        marker=MK[c],
        alpha=0.9,
        edgecolor="white",
        lw=1,
    )

from matplotlib.lines import Line2D

a1.legend(
    handles=[
        Line2D(
            [],
            [],
            marker=MK[c],
            ls="",
            color=ColourCodes[c],
            markersize=13,
            markeredgecolor="white",
            label=f"{c} (n={(dataframe['kmean_result'] == c).sum()})",
        )
        for c in ORDER
    ],
    frameon=False,
    ncol=2,
    fontsize=12,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    columnspacing=1.2,
    handletextpad=0.3,
)

a1.axhline(0, c="k", lw=0.8, ls=":")
a1.axvline(0, c="k", lw=0.8, ls=":")
a1.set_xlabel("xG difference (team - opponent)")
a1.set_ylabel("Possession difference (pp)")
a1.set_title("1  Case 1 · Four match profiles (k=4)")

Text(0.5, 1.0, '1  Case 1 · Four match profiles (k=4)')

In [ ]:
zc = (center - dataframe[F1].mean()) / dataframe[F1].std()                          # 2. centroid heatmap
a2.imshow(zc.values, cmap="RdBu", vmin=-2.2, vmax=2.2, aspect="auto")
a2.set_xticks(range(4)); a2.set_xticklabels(["xG\ndiff", "Possession\ndiff", "Shots on\ntarget diff" if third_performance_variable == "sot" else "Shots\ndiff", "Goal\ndiff"], fontsize=12)
a2.set_yticks(range(4)); a2.set_yticklabels(ORDER, fontsize=13)
for i in range(4):
    for j in range(4): a2.text(j, i, f"{center.values[i, j]:+.1f}", ha="center", va="center", fontsize=15, fontweight="bold", color="white" if abs(zc.values[i, j]) > .9 else "black")
a2.set_title("2  What defines each profile (average values)"); a2.spines[:].set_visible(False); a2.tick_params(length=0) 

In [ ]:
bot = np.zeros(4)                                                 # 3. outcomes per class
for k in ["Win", "Draw", "Loss"]:
    a3.barh(range(4), comp[k].values, left=bot, color=OUT[k], edgecolor="white", lw=2, label=k)
    for i in range(4):
        if comp[k].values[i] > .07: a3.text(bot[i] + comp[k].values[i] / 2, i, f"{comp[k].values[i]:.0%}", ha="center", va="center", color="white", fontsize=13 if comp[k].values[i] > .15 else 11, fontweight="bold")
    bot += comp[k].values
a3.set_yticks(range(4)); a3.set_yticklabels([f"{c}\n(n={int(comp.loc[c,'n'])})" for c in ORDER], fontsize=12); a3.invert_yaxis(); a3.set_xlim(0, 1)
a3.set_xlabel("Share of results"); a3.set_title("3  Results inside each profile")
a3.legend(frameon=False, ncol=3, fontsize=13, loc="upper center", bbox_to_anchor=(.45, -.2))
 
g = home_perspective[home_perspective.case2 == "Expected"]; a4.scatter(g.perf_idx, g.goal_delta, s=75, c=EXPC, marker="o", alpha=.75, edgecolor="white", lw=.8)
g = home_perspective[home_perspective.case2 == "Unexpected"]; a4.scatter(g.perf_idx, g.goal_delta, s=100, c=UNXC, marker="D", edgecolor="black", lw=1.5, zorder=5)
a4.axhline(0, c="k", lw=.8, ls=":"); a4.axvline(0, c="k", lw=.8, ls=":"); a4.set_xlim(-2.3, 3.4); a4.set_ylim(-5, 8)
for (tx, ty, t_) in [(.97, .97, "Better side won"), (.03, .03, "Better side won"), (.03, .97, "Better side did not win"), (.97, .03, "Better side did not win")]:
    a4.text(tx, ty, t_, transform=a4.transAxes, ha="right" if tx > .5 else "left", va="top" if ty > .5 else "bottom", fontsize=12, color="#444", style="italic")
a4.set_xlabel("Performance index (xG, possession, " + ("shots on target" if third_performance_variable == "sot" else "shots") + "; home - away, in SD)"); a4.set_ylabel("Goal difference (home - away)")
a4.set_title("4  Case 2 · Was the result expected? (k=2)"); a4.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, .86), fontsize=13)

/tmp/ipykernel_1067/113794017.py:17: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a4.set_title("4  Case 2 · Was the result expected? (k=2)"); a4.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, .86), fontsize=13)


In [ ]:
tp = top.iloc[::-1]                                               # 5. top 10 unexpected results (horizontal bars)
a5.barh(range(10), tp.surprise, color=UNXC, edgecolor="black", lw=1)
a5.set_yticks(range(10)); a5.set_yticklabels(tp.label, fontsize=13)
for i, v in enumerate(tp.surprise): a5.text(v + .03, i, f"{v:.1f}", va="center", fontsize=13, fontweight="bold")
a5.set_xlim(0, tp.surprise.max() * 1.12)
a5.set_xlabel("Performance index of the better side that failed to win (SD)"); a5.set_title("5  Ten most unexpected results")
fig.suptitle("Beyond the scoreboard: do xG and possession describe the result?  ·  FIFA World Cup 2026", fontsize=21, fontweight="bold", y=.98)
fig.savefig("/work/" + "beyond_the_scoreboard_dashboard.png", dpi=150, bbox_inches="tight", facecolor="white"); plt.close(fig)
print(top[["label", "perf_idx", "goal_normalized", "surprise", "case2"]].round(2).to_string())

                             label  perf_idx  goal_normalized  surprise       case2
33             Ecuador 0–0 Curaçao      1.81             0.00      1.81  Unexpected
12            Spain 0–0 Cabo Verde      1.59             0.00      1.59  Unexpected
14        Saudi Arabia 1–1 Uruguay     -1.27             0.00      1.27  Unexpected
36             Belgium 0–0 IR Iran      1.08             0.00      1.08  Unexpected
75  Netherlands 1–1 Morocco (pens)     -1.04             0.00      1.04  Unexpected
31            Türkiye 0–1 Paraguay      1.01            -0.48      1.01  Unexpected
74     Germany 1–1 Paraguay (pens)      1.00             0.00      1.00  Unexpected
4            Qatar 1–1 Switzerland     -0.95             0.00      0.95  Unexpected
39          Uruguay 2–2 Cabo Verde      0.76             0.00      0.76  Unexpected
20           Portugal 1–1 Congo DR      0.71             0.00      0.71  Unexpected


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f1104627-0aca-40f0-9d55-4d87c60a8049' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>